# Semantic Segmentation: Model Comparison on Cityscapes Validation Dataset

Compares ERFNet, pretrained EoMT (Cityscapes + COCO), and the COCO-pretrained EoMT then finetuned on CitsScapes

**Sections**
1. Demo images - qualitative comparison on a single validation sample
2. Quantitative evaluation - `results_semantic/comparison.csv`

## Setup

In [ ]:
import os, sys
from huggingface_hub import snapshot_download
try:
    IN_COLAB = 'google.colab' in str(get_ipython())
except NameError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    REPO_URL = 'https://github.com/timmfy/MaskArchitectureAnomaly_CourseProject.git'
    REPO_DIR = '/content/MaskArchitectureAnomaly_CourseProject'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    from google.colab import drive
    drive.mount('/content/drive') 
else:
    REPO_DIR = os.path.abspath('.')

for _p in [REPO_DIR, os.path.join(REPO_DIR, 'eval')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

if not os.path.exists(os.path.join(REPO_DIR, 'weights')):
    snapshot_download(
        repo_id="timmfy/comprehensive-road-scene-understanding", 
        local_dir="./weights",
        allow_patterns=["*.pth", "*.bin", "*.ckpt"],
    )

In [ ]:
###### Edit the paths below as needed ######

DATASET_DIR = os.path.join(REPO_DIR, 'datasets/cityscapes')
WEIGHTS_DIR = os.path.join(REPO_DIR, 'weights/')
FINETUNED_DIR  = os.path.join(REPO_DIR, 'weights/')

W_ERFNET = os.path.join(WEIGHTS_DIR, 'erfnet_pretrained.pth')
W_CITYSCAPES = os.path.join(WEIGHTS_DIR, 'eomt_cityscapes.bin')
W_COCO = os.path.join(WEIGHTS_DIR, 'eomt_coco.bin')
W_COCO_FINETUNED = os.path.join(FINETUNED_DIR, 'eomt_coco_finetuned.ckpt')

CFG_CS = os.path.join(REPO_DIR, 'eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml')
CFG_COCO = os.path.join(REPO_DIR, 'eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml')
CFG_FT = os.path.join(REPO_DIR, 'eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x_finetuning.yaml')

IGNORE_INDEX = 255
IMG_IDX      = 0   # which validation image to use for demos

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from eomt_tools import eomt_setup, eomt_inference
device = eomt_setup.setup_environment(eomt_path=os.path.join(REPO_DIR, 'eomt'))
print('Device:', device)

CITYSCAPES_CLASSES = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
    'traffic_light', 'traffic_sign', 'vegetation', 'terrain',
    'sky', 'person', 'rider', 'car', 'truck', 'bus', 'train',
    'motorcycle', 'bicycle'
]


## 1. Demo Images

All models are shown on the same Cityscapes validation image (index `IMG_IDX`).

### 1.1 EoMT Cityscapes-Pretrained Semantic Segmentation

In [ ]:
cfg_cs   = eomt_setup.load_config(CFG_CS)
data_cs  = eomt_setup.setup_data(cfg_cs, data_path=DATASET_DIR)
model_cs = eomt_setup.load_model(cfg_cs, data_cs, device, weights_path=W_CITYSCAPES)

demo_img, demo_target = data_cs.val_dataloader().dataset[IMG_IDX]
pred_cs, target_cs = eomt_inference.infer_semantic(
    model_cs, demo_img, demo_target, device, data_cs.img_size, IGNORE_INDEX
)
eomt_inference.plot_semantic_results(demo_img, pred_cs, target_cs, 'EoMT Cityscapes Pretrained — Semantic', IGNORE_INDEX)

### 1.2 EoMT COCO-Pretrained Panoptic Segmentation

The COCO-pretrained model performs panoptic segmentation: semantic class + instance ID per pixel.
Black borders delineate individual instances within the same semantic class. In addition, after the inference, the predicted COCO labels are mapped to the corresponding ones of the Cityscapes to obtain the consistent color mapping for visual comparison

In [ ]:
cfg_coco  = eomt_setup.load_config(CFG_COCO)
data_coco = eomt_setup.setup_data(cfg_coco, data_path=DATASET_DIR)
coco_img, coco_target = data_coco.val_dataloader().dataset[IMG_IDX]

model_coco = eomt_setup.load_model(cfg_coco, data_coco, torch.device('cpu'),
                                    weights_path=W_COCO)
model_coco = model_coco.to(device=device, dtype=torch.float32)

sem_pan, inst_pan, sem_tgt_pan, inst_tgt = eomt_inference.infer_panoptic(
    model_coco, coco_img, coco_target, device
)
sem_pan_cs = eomt_inference.map_coco_preds_to_cityscapes(sem_pan)
eomt_inference.plot_panoptic_results(
    model_coco, coco_img, sem_pan_cs, inst_pan, sem_tgt_pan, inst_tgt, "EoMT COCO Pretrained — Panoptic"
)

### 1.3 EoMT COCO-Pretrained Semantic Segmentation

Same model as 1.2, run in semantic-only mode (no instance separation).

In [ ]:
sem_pred_coco, sem_tgt_coco = eomt_inference.infer_semantic(
    model_coco, coco_img, coco_target, device, data_coco.img_size, IGNORE_INDEX
)
sem_pred_coco_cs = eomt_inference.map_coco_preds_to_cityscapes(sem_pred_coco)
eomt_inference.plot_semantic_results(coco_img, sem_pred_coco_cs, sem_tgt_coco, "EoMT COCO Pretrained — Semantic", IGNORE_INDEX)

### 1.4 EoMT COCO after Finetuning: Semantic Segmentation

The EoMT COCO model finetuned on Cityscapes with the backbone unfrozen after 10 epochs and positional embeddings interpolated

In [ ]:
cfg_ft   = eomt_setup.load_config(CFG_FT)
data_ft  = eomt_setup.setup_data(cfg_ft, data_path=DATASET_DIR)
ft_img, ft_target = data_ft.val_dataloader().dataset[IMG_IDX]
m = eomt_setup.load_model(cfg_ft, data_ft, device, weights_path=W_COCO_FINETUNED)
pred_ft, tgt_ft = eomt_inference.infer_semantic(
    m, ft_img, ft_target, device, data_ft.img_size, IGNORE_INDEX
)
eomt_inference.plot_semantic_results(ft_img, pred_ft, tgt_ft, f'EoMT COCOFinetuned — Semantic', IGNORE_INDEX)

## 2. Quantitative Comparison

Evaluates all 3 EoMT models + the ERFNet pixel-based baseline on the Cityscapes validation set.

In [ ]:
EVAL_LIMIT = 1  # None = full 500-image val set

from eval.semantic_eval import evaluate_model, write_results_csv, print_results_summary, CITYSCAPES_CLASSES

results = []
results.append(evaluate_model("ERFNet", W_ERFNET, None, DATASET_DIR, device, "erfnet", EVAL_LIMIT, IGNORE_INDEX))
results.append(evaluate_model("EoMT Cityscapes Pretrained", W_CITYSCAPES, CFG_CS, DATASET_DIR, device, "eomt_cityscapes", EVAL_LIMIT, IGNORE_INDEX))
results.append(evaluate_model("EoMT COCO Pretrained", W_COCO, CFG_COCO, DATASET_DIR, device, "eomt_coco", EVAL_LIMIT, IGNORE_INDEX))
results.append(evaluate_model("EoMT Finetuned Interp", W_COCO_FINETUNED, CFG_FT, DATASET_DIR, device, "eomt_finetuned", EVAL_LIMIT, IGNORE_INDEX))

csv_path = os.path.join(REPO_DIR, 'results_semantic', 'comparison.csv')
write_results_csv(results, csv_path, CITYSCAPES_CLASSES)
print_results_summary(results, CITYSCAPES_CLASSES)